# Full-Parameter Fine-Tuning Tutorial (Domain Specific) 🚀

**Is notebook mein hum seekhenge:**
1. Apne custom domain dataset (Metformin drug PDF) se text data extract aur preprocess kaise karte hain.
2. Pre-trained **TinyLlama-1.1B** model ko pure parameters ke saath fine-tune (Full Fine-tuning) kaise karte hain.
3. Training settings ko Apple Silicon (Mac MPS) par numerical errors (jaise NaNs) se bachane ke liye kaise optimize karte hain.

*Note: Ye ek non-instruction domain specific pre-training type task hai.*

In [1]:
# Sabse pehle training ke liye saare required libraries install karte hain
!uv pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF --quiet

In [2]:
from datasets import Dataset, load_dataset

## 1. Dataset Preparation (डेटासेट की तैयारी) 📂

Pehle hum training data load karenge. Hum Hugging Face hub se pre-built dataset use kar sakte hain, ya fir local PDF documents se text extract kar sakte hain.

In [3]:
# Hugging Face library se standard datasets load karne ke options (commented)
# dataset = load_dataset("HuggingFaceFW/fineweb")
# pubmed = load_dataset("ncbi/pubmed")
# dataset = load_dataset("datajuicer/the-pile-oubmed-abstracts-refined-by-data-juicer")
# dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")

# owt = load_dataset("Skylion007/openwebtext")
# ds = load_dataset("armanc/scientific_papers")


### Option A: Loading Dataset from Hugging Face Hub 🌐

Hum test ke liye standard Hugging Face dataset (jaise TinyStories) load kar sakte hain.

In [4]:
# TinyStories dataset ko sample check ke liye download karte hain
dataset = load_dataset("roneneldan/TinyStories", split="train")

In [5]:
# Downloaded dataset ka shape aur sample entries print karte hain
print(dataset)
print(dataset[0])
print(dataset[-1])

Dataset({
    features: ['text'],
    num_rows: 2119719
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
{'text': 'Once upon a time, there was an adorable little cat named Kitty. Kitty loved to polish her toy car with a soft cloth. One sunny day, she decided to take her shiny car to the park.\n\nAt the park, she met a friendl

### Option B: Custom PDF Domain Data Extraction 📄

Hum medical field ka model banana chahte hain. Iske liye `Metformin.pdf` file se paragraphs extract karenge using `PyMuPDF` (`fitz`).

In [6]:
# PDF file se plain text extract karne ka utility function
import fitz

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    
    return text_blocks


pdf_texts = extract_text_from_pdf("Metformin.pdf")

print(pdf_texts)
print(len(pdf_texts[0]))

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [7]:
# Extract kiya gaya raw text bada ho sakta hai. Hum paragraphs ko sensible chunks (length > 30) mein split karte hain
import re

def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:
                paragraphs.append(clean)
    
    return paragraphs

In [8]:
# PDF blocks ko split karke list of paragraphs mein store karte hain
paragraphs = split_paragraphs(pdf_texts)
paragraphs

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA redu

In [9]:
# Har paragraph ko HF Dataset standard schema format [{'text': '...'}] mein parse karte hain
data = [{"text": p} for p in paragraphs]
data

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

In [10]:
# Dictionary data list se local Hugging Face Dataset instance ready karte hain
dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

## 2. Model Selection & Tokenization (मॉडल और टोकनाइजेशन) 🤖

Hum `TinyLlama-1.1B` select kar rahe hain kyunki ye lightweight hai aur local macOS CPU/GPU (MPS) memory mein efficiently train ho jata hai.

In [11]:
# Target base LLM model ID define karte hain
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [12]:
# Fine-tuning ke liye essential trainer modules, classes aur collators import karte hain
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [13]:
# Model ke corresponding pre-trained tokenizer load karte hain
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [14]:
# Check karte hain tokenizer ke paas predefined Pad aur EOS tokens hain ya nahi
print(tokenizer.eos_token)
print(tokenizer.pad_token)

</s>
None


In [15]:
# Agar padding token defined nahi hai, to standard EOS (End of Sentence) token ko hi pad token assign kar dete hain
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Dataset tokenize function: input_ids copy karke labels array mein fill karte hain self-supervised training ke liye
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [17]:
# Puray dataset par tokenization apply karte hain aur raw text column ko remove karte hain
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [18]:
# Pre-trained TinyLlama model load karte hain full precision mode mein
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Training configuration define karte hain. bf16=True use karenge for Apple Silicon/MPS numerical stability (taaki NaN error na aaye)
training_args = TrainingArguments(
    output_dir="./../llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    bf16=True,
    report_to="none"
)

In [20]:
# Training configuration define karte hain. bf16=True use karenge for Apple Silicon/MPS numerical stability (taaki NaN error na aaye)
# help(TrainingArguments)

In [21]:
# Trainer setup: model, data collator aur tokenized training dataset pass karte hain
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
# Model training start karne ke liye is code ko run karein
# trainer.train()

/Users/kevin/Desktop/engineer/LLMs-from-scratch/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 3816890880 vs 3816890768